In [3]:
import cv2
import numpy as np
import os
import requests
import matplotlib.image as mpimg

In [14]:
import cv2
import numpy as np

class YOLOv3Pipeline:
    def __init__(self, config_path, weights_path, names_path):
        self.net = cv2.dnn.readNet(weights_path, config_path)
        
        with open(names_path, 'r') as f:
            self.classes = [line.strip() for line in f.readlines()]

        self.output_layers = self.net.getUnconnectedOutLayersNames()
        self.colors = np.random.uniform(0, 255, size=(len(self.classes), 3))

    def process_image(self, img_path):
        img = cv2.imread(img_path)
        height, width, channels = img.shape

        blob = cv2.dnn.blobFromImage(img, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
        self.net.setInput(blob)
        outs = self.net.forward(self.output_layers)
        boxes, confidences, class_ids = self._process_detections(outs, width, height)
        self._draw_boxes(img, boxes, confidences, class_ids)
        
        return img

    def _process_detections(self, outs, width, height):
        boxes = []
        confidences = []
        class_ids = []
        
        for out in outs:
            for detection in out:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                
                if confidence > 0.5:
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)

                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)
        
        return boxes, confidences, class_ids

    def _draw_boxes(self, img, boxes, confidences, class_ids):
        indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

        font = cv2.FONT_HERSHEY_PLAIN
        
        for i in range(len(boxes)):
            if i in indexes:
                x, y, w, h = boxes[i]
                label = str(self.classes[class_ids[i]])
                color = self.colors[class_ids[i]]
                cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
                cv2.putText(img, f'{label} {confidences[i]:.2f}', (x, y + 30), font, 3, color, 3)

    def save_image(self, img, output_path):
        cv2.imwrite(output_path, img)

if __name__ == '__main__':
    config_path = 'yolov3.cfg'
    weights_path = 'yolov3.weights'
    names_path = 'coco.names'
    img_path = 'cat.jpg'
    output_image_path = 'detected_cat.jpg'

    yolo_pipeline = YOLOv3Pipeline(config_path, weights_path, names_path)
    result_img = yolo_pipeline.process_image(img_path)
    yolo_pipeline.save_image(result_img, output_image_path)